
# Economic Prosperity, Environment & Social Outcomes  


**Goal.** Explore whether countries with **higher GDP per capita** emit **more CO₂ per capita**, and extend the analysis with a **third indicator** (I use *Life Expectancy*).  
We'll follow a clean pipeline: **Load → Clean → Merge → Feature engineer → Analyze → Visualize → Interpret**.

> Datasets (Our World in Data / World Bank via OWID):  
> • **CO₂ emissions per capita** (tonnes): `https://ourworldindata.org/grapher/co-emissions-per-capita.csv`  
> • **GDP per capita (constant USD)**: `https://ourworldindata.org/grapher/gdp-per-capita-worldbank-constant-usd.csv`  
> • **Life expectancy at birth (years)**: `https://ourworldindata.org/grapher/life-expectancy.csv`

I save **publication‑quality PNGs** for every figure in a `figs/` folder so they’re easy to drop into reports.


## 1) Imports

In [ ]:

# Core stack
import os, io, math, sys, warnings
import numpy as np
import pandas as pd

# Stats & viz
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# A tiny helper so figures look consistent
sns.set(context="talk", style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["savefig.dpi"] = 150

# Folder for all exported images
FIG_DIR = "figs"
os.makedirs(FIG_DIR, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
warnings.filterwarnings("ignore")

print("Ready ✅")


## 2) Load datasets & quick sanity check

In [ ]:

CO2_URL = "https://ourworldindata.org/grapher/co-emissions-per-capita.csv"
GDP_URL = "https://ourworldindata.org/grapher/gdp-per-capita-worldbank-constant-usd.csv"
LE_URL  = "https://ourworldindata.org/grapher/life-expectancy.csv"

# I keep names compact and consistent
co2 = pd.read_csv(CO2_URL)
gdp = pd.read_csv(GDP_URL)
life = pd.read_csv(LE_URL)

for df, name in [(co2,"co2"), (gdp,"gdp"), (life,"life")]:
    print(name, df.shape, list(df.columns[:5]))

# These OWID CSVs use columns: 'Entity','Code','Year','<metric>'
co2.rename(columns={"Entity":"Country", "co-emissions-per-capita":"CO2_pc"}, inplace=True)
gdp.rename(columns={"Entity":"Country", "gdp-per-capita-worldbank-constant-usd":"GDP_pc"}, inplace=True)
life.rename(columns={"Entity":"Country", "life-expectancy":"LifeExp"}, inplace=True)

co2.head(3)


## 3) Clean & standardize
- Standardize country names if needed
- Keep overlapping years only
- Drop rows with obvious nulls

In [ ]:

# Keep country rows (drop aggregates like 'World', 'Asia', etc. by requiring ISO 'Code' to be 3 letters)
def is_country(row):
    code = str(row.get("Code", ""))
    return len(code) == 3 and code.isalpha()

co2 = co2[co2.apply(is_country, axis=1)].copy()
gdp = gdp[gdp.apply(is_country, axis=1)].copy()
life = life[life.apply(is_country, axis=1)].copy()

# Restrict to overlapping years
years_intersection = sorted(set(co2["Year"]).intersection(set(gdp["Year"])).intersection(set(life["Year"])))
co2 = co2[co2["Year"].isin(years_intersection)]
gdp = gdp[gdp["Year"].isin(years_intersection)]
life = life[life["Year"].isin(years_intersection)]

# Drop nulls in key metrics
co2 = co2.dropna(subset=["Annual CO₂ emissions (per capita)"])
gdp = gdp.dropna(subset=["GDP per capita (constant 2015 US$)"])
life = life.dropna(subset=["Period life expectancy at birth"])

co2.shape, gdp.shape, life.shape


## 4) Merge on `Country` + `Year`

In [ ]:

# Merge step-by-step to keep it readable
df = pd.merge(gdp[["Country","Code","Year","GDP per capita (constant 2015 US$)"]],
              co2[["Country","Code","Year","Annual CO₂ emissions (per capita)"]],
              on=["Country","Code","Year"],
              how="inner")

df = pd.merge(df, life[["Country","Code","Year","Period life expectancy at birth"]],
              on=["Country","Code","Year"], how="inner")

print(df.shape)
df.sample(5, random_state=42)


## 5) Feature engineering: GDP bands
Low (< $5k), Medium ($5k–$15k), High (> $15k) — using constant USD GDP per capita

In [ ]:

def gdp_band(x):
    if x < 5_000:
        return "Low"
    elif x <= 15_000:
        return "Medium"
    else:
        return "High"

df["GDP_Label"] = df["GDP per capita (constant 2015 US$)"].apply(gdp_band)
df["GDP_Label"] = pd.Categorical(df["GDP_Label"], categories=["Low","Medium","High"], ordered=True)
df.head()


## 6) Descriptive analytics (Mean, SEM, 95% CI) by GDP band & Year

In [ ]:

# helper: standard error of the mean
def sem(x):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) / (len(x) ** 0.5) if len(x) > 1 else np.nan

grp = (df.groupby(["GDP_Label","Year"])
         .agg(mean_CO2=("Annual CO₂ emissions (per capita)","mean"),
              sem_CO2=("Annual CO₂ emissions (per capita)", sem))
         .reset_index())
grp["ci_low"] = grp["mean_CO2"] - 1.96*grp["sem_CO2"]
grp["ci_high"]= grp["mean_CO2"] + 1.96*grp["sem_CO2"]

grp.head()


## 7) Visualization: CO₂ per capita over time by GDP band (95% CI shaded)

In [ ]:

fig, ax = plt.subplots()
for label, sub in grp.groupby("GDP_Label"):
    ax.plot(sub["Year"], sub["mean_CO2"], label=label)
    ax.fill_between(sub["Year"], sub["ci_low"], sub["ci_high"], alpha=0.2)

ax.set_title("CO₂ Emissions per Capita by GDP Band")
ax.set_ylabel("Tonnes per person")
ax.legend(title="GDP Band")
plt.tight_layout()
png1 = os.path.join(FIG_DIR, "co2_by_gdpband_ci.png")
plt.savefig(png1); plt.show()
print("Saved:", png1)


## 8) Scatter: GDP per capita vs CO₂ per capita (+Pearson r)

In [ ]:

# Use the latest common year to minimize panel effects in the scatter
latest_year = df["Year"].max()
snap = df[df["Year"]==latest_year].copy()
r, p = stats.pearsonr(snap["GDP per capita (constant 2015 US$)"], snap["Annual CO₂ emissions (per capita)"])

sns.scatterplot(data=snap, x="GDP per capita (constant 2015 US$)", y="Annual CO₂ emissions (per capita)", hue="GDP_Label")
plt.xscale("log")  # GDP per capita is skewed; log scale is standard
plt.title(f"GDP per capita vs CO₂ per capita — {latest_year}\nPearson r={r:.2f}, p={p:.3g}")
plt.tight_layout()
png2 = os.path.join(FIG_DIR, "scatter_gdp_vs_co2.png")
plt.savefig(png2); plt.show()
print("Saved:", png2)



## 9) New hypothesis (Part 2)
**H₁:** *Countries with higher CO₂ emissions per capita have lower life expectancy, after accounting for GDP band.*  
We'll test the association between **CO₂ per capita** and **Life Expectancy**, and visualize the relationship.


In [ ]:

# Correlation within each GDP band (same latest year snapshot)
rows = []
for label, sub in snap.groupby("GDP_Label"):
    if len(sub) > 2:
        r, p = stats.pearsonr(sub["Annual CO₂ emissions (per capita)"], sub["Period life expectancy at birth"])
        rows.append({"GDP_Label":label, "r_CO2_LifeExp":r, "p_value":p, "n":len(sub)})
band_corr = pd.DataFrame(rows)
band_corr


In [ ]:

# Scatter with lowess trend (Seaborn's regplot with lowess)
sns.lmplot(data=snap, x="Annual CO₂ emissions (per capita)", y="Period life expectancy at birth", hue="GDP_Label", lowess=True, scatter_kws=dict(alpha=0.6))
plt.title(f"Life Expectancy vs CO₂ per capita — {latest_year} (LOWESS trend)")
plt.tight_layout()
png3 = os.path.join(FIG_DIR, "lifeexp_vs_co2_lowess.png")
plt.savefig(png3);
plt.show()
print("Saved:", png3)


## 10) Bonus: Life expectancy over time by GDP band (95% CI)

In [ ]:

grp_le = (df.groupby(["GDP_Label","Year"])
            .agg(mean_LE=("Period life expectancy at birth","mean"),
                 sem_LE=("Period life expectancy at birth", lambda x: pd.Series(x).std(ddof=1)/ (len(x)**0.5)))
            .reset_index())
grp_le["ci_low"] = grp_le["mean_LE"] - 1.96*grp_le["sem_LE"]
grp_le["ci_high"]= grp_le["mean_LE"] + 1.96*grp_le["sem_LE"]

fig, ax = plt.subplots()
for label, sub in grp_le.groupby("GDP_Label"):
    ax.plot(sub["Year"], sub["mean_LE"], label=label)
    ax.fill_between(sub["Year"], sub["ci_low"], sub["ci_high"], alpha=0.2)

ax.set_title("Life Expectancy by GDP Band")
ax.set_ylabel("Years")
ax.legend(title="GDP Band")
plt.tight_layout()
png4 = os.path.join(FIG_DIR, "lifeexp_by_gdpband_ci.png")
plt.savefig(png4); plt.show()
print("Saved:", png4)



## 11) Interpretation

- **Part 1 (Core)**: As GDP per capita increases, *average* CO₂ emissions per person are generally **higher** (see the line plot and the scatter).  
  The shaded 95% CIs are reasonably tight for High‑income groups and wider for Low‑income groups, reflecting greater variability and smaller sample sizes.

- **Part 2 (New indicator)**: For the latest year snapshot, **Life Expectancy** tends to be **higher** in wealthier countries and **negatively associated** with CO₂ per capita **within** some GDP bands (see the LOWESS trends).  
  This suggests environmental impacts may correlate with health outcomes, but GDP level and regional composition matter — correlation ≠ causation.

**Reliability note.** CIs use `mean ± 1.96 × SEM`. Results depend on data completeness and the quality of country‑level reporting.


## 12) Appendix: export cleaned panel (optional)

In [ ]:

out_csv = "clean_country_panel.csv"
df.to_csv(out_csv, index=False)
print("Wrote:", out_csv, "→", pd.read_csv(out_csv).shape)
